# Training Model Klasifikasi Dokumen Surat
## Penerapan Naive Bayes + TF-IDF

**Peneliti:** Rivaldo Janter Tampubolon (221011402289)
**Sistem:** Pengarsipan Dokumen Surat PT Almex Bintang Timur
**Algoritma:** Multinomial Naive Bayes + TF-IDF

In [ ]:
# 1. IMPORT LIBRARY
import re, string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score)
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
print('Library berhasil diimport!')

## 2. Dataset
Dataset simulasi dokumen surat PT Almex Bintang Timur. 70 dokumen, 7 kategori jenis, 2 arah.

In [ ]:
data = [
    # PURCHASE ORDER
    {'text': 'Surat Pesanan Purchase Order Nomor 001 PO ABT Januari 2025 pemesanan barang laptop printer kebutuhan kantor pusat', 'arah': 'Masuk', 'jenis': 'Purchase Order'},
    {'text': 'Purchase Order Nomor 002 PO ABT Februari 2025 pesanan pengadaan server jaringan proyek pengembangan sistem informasi', 'arah': 'Masuk', 'jenis': 'Purchase Order'},
    {'text': 'Surat Pesanan Nomor 003 PO ABT Maret 2025 pemesanan material kantor kertas tinta alat tulis kantor jumlah besar', 'arah': 'Masuk', 'jenis': 'Purchase Order'},
    {'text': 'Purchase Order Pengadaan Barang Nomor 004 PO ABT April 2025 pesanan perangkat komputer desktop divisi akuntansi keuangan', 'arah': 'Masuk', 'jenis': 'Purchase Order'},
    {'text': 'PO Nomor 005 ABT Mei 2025 pemesanan furniture kantor meja kerja kursi ergonomis lemari arsip renovasi gedung', 'arah': 'Masuk', 'jenis': 'Purchase Order'},
    {'text': 'Purchase Order Nomor 006 ABT Juni 2025 pengadaan software lisensi Microsoft Office antivirus karyawan', 'arah': 'Masuk', 'jenis': 'Purchase Order'},
    {'text': 'Surat Pesanan Barang Nomor 007 PO ABT Juli 2025 pemesanan spare part kendaraan operasional perusahaan pemeliharaan armada', 'arah': 'Masuk', 'jenis': 'Purchase Order'},
    {'text': 'Purchase Order Pengadaan Nomor 008 ABT Agustus 2025 pesanan alat laboratorium bahan kimia divisi quality control', 'arah': 'Masuk', 'jenis': 'Purchase Order'},
    {'text': 'PO Nomor 009 ABT September 2025 pemesanan mesin produksi peralatan pabrik lini manufaktur baru', 'arah': 'Masuk', 'jenis': 'Purchase Order'},
    {'text': 'Purchase Order Nomor 010 ABT Oktober 2025 pengadaan seragam kerja alat pelindung diri karyawan gudang', 'arah': 'Masuk', 'jenis': 'Purchase Order'},
    # INVOICE
    {'text': 'Invoice Nomor INV ABT 2025 001 tagihan pembayaran jasa konsultasi manajemen bulan Januari lima puluh juta rupiah', 'arah': 'Masuk', 'jenis': 'Invoice'},
    {'text': 'Faktur Invoice Nomor INV ABT 2025 002 tagihan pengadaan laptop komputer desktop sesuai purchase order', 'arah': 'Masuk', 'jenis': 'Invoice'},
    {'text': 'Invoice Pembayaran Nomor INV ABT 2025 003 tagihan jasa maintenance server bulanan periode Februari Maret', 'arah': 'Masuk', 'jenis': 'Invoice'},
    {'text': 'Faktur Tagihan Nomor INV ABT 2025 004 invoice jasa pengiriman logistik distribusi barang cabang', 'arah': 'Masuk', 'jenis': 'Invoice'},
    {'text': 'Invoice Nomor INV ABT 2025 005 tagihan sewa gedung kantor pusat triwulan pertama tahun 2025', 'arah': 'Masuk', 'jenis': 'Invoice'},
    {'text': 'Faktur Invoice Pembelian Nomor INV ABT 2025 006 tagihan bahan baku produksi supplier utama perusahaan', 'arah': 'Masuk', 'jenis': 'Invoice'},
    {'text': 'Invoice Jasa Konsultan Nomor INV ABT 2025 007 tagihan konsultasi hukum perpajakan semester satu', 'arah': 'Masuk', 'jenis': 'Invoice'},
    {'text': 'Tagihan Invoice Nomor INV ABT 2025 008 pembayaran jasa cleaning service keamanan gedung bulanan', 'arah': 'Masuk', 'jenis': 'Invoice'},
    {'text': 'Invoice Nomor INV ABT 2025 009 tagihan langganan internet telepon kantor periode Januari Juni', 'arah': 'Masuk', 'jenis': 'Invoice'},
    {'text': 'Faktur Invoice Nomor INV ABT 2025 010 tagihan cetak dokumen percetakan kebutuhan promosi perusahaan', 'arah': 'Masuk', 'jenis': 'Invoice'},
    # SURAT PENAWARAN
    {'text': 'Surat Penawaran Kerjasama Nomor SP 001 ABT 2025 PT Mitra Solusi menawarkan kerjasama pengembangan sistem informasi', 'arah': 'Masuk', 'jenis': 'Surat Penawaran'},
    {'text': 'Surat Penawaran Harga Nomor SP 002 ABT 2025 penawaran harga pengadaan perangkat keras komputer jaringan', 'arah': 'Masuk', 'jenis': 'Surat Penawaran'},
    {'text': 'Penawaran Kerjasama Bisnis Nomor SP 003 ABT 2025 proposal kerjasama pemasaran produk wilayah Jawa', 'arah': 'Masuk', 'jenis': 'Surat Penawaran'},
    {'text': 'Surat Penawaran Jasa Nomor SP 004 ABT 2025 penawaran jasa konsultasi IT pengembangan aplikasi web perusahaan', 'arah': 'Masuk', 'jenis': 'Surat Penawaran'},
    {'text': 'Penawaran Pengadaan Barang Nomor SP 005 ABT 2025 penawaran harga pengadaan alat kantor furniture', 'arah': 'Masuk', 'jenis': 'Surat Penawaran'},
    {'text': 'Surat Penawaran Tender Nomor SP 006 ABT 2025 penawaran tender proyek pembangunan infrastruktur jaringan kantor', 'arah': 'Masuk', 'jenis': 'Surat Penawaran'},
    {'text': 'Penawaran Layanan Cloud Nomor SP 007 ABT 2025 penawaran layanan cloud hosting backup data perusahaan', 'arah': 'Masuk', 'jenis': 'Surat Penawaran'},
    {'text': 'Surat Penawaran Training Nomor SP 008 ABT 2025 penawaran pelatihan karyawan peningkatan kompetensi IT', 'arah': 'Masuk', 'jenis': 'Surat Penawaran'},
    {'text': 'Penawaran Sewa Kendaraan Nomor SP 009 ABT 2025 penawaran sewa kendaraan operasional kebutuhan logistik', 'arah': 'Masuk', 'jenis': 'Surat Penawaran'},
    {'text': 'Surat Penawaran Asuransi Nomor SP 010 ABT 2025 penawaran asuransi kesehatan jiwa seluruh karyawan', 'arah': 'Masuk', 'jenis': 'Surat Penawaran'},
    # KONTRAK
    {'text': 'Kontrak Kerjasama Nomor KTR 001 ABT 2025 perjanjian kerjasama PT Almex PT Mitra Solusi proyek pengembangan sistem', 'arah': 'Masuk', 'jenis': 'Kontrak'},
    {'text': 'Perjanjian Kontrak Nomor KTR 002 ABT 2025 kontrak pengadaan barang jasa nilai kontrak lima ratus juta rupiah', 'arah': 'Masuk', 'jenis': 'Kontrak'},
    {'text': 'Kontrak Kerja Nomor KTR 003 ABT 2025 perjanjian kerja karyawan tetap posisi staff IT administrasi', 'arah': 'Keluar', 'jenis': 'Kontrak'},
    {'text': 'Kontrak Sewa Nomor KTR 004 ABT 2025 perjanjian sewa gedung kantor jangka waktu dua tahun ke depan', 'arah': 'Masuk', 'jenis': 'Kontrak'},
    {'text': 'Perjanjian Kontrak Layanan Nomor KTR 005 ABT 2025 kontrak maintenance support sistem informasi perusahaan', 'arah': 'Masuk', 'jenis': 'Kontrak'},
    {'text': 'Kontrak Kemitraan Nomor KTR 006 ABT 2025 perjanjian kemitraan strategis pengembangan bisnis bersama', 'arah': 'Keluar', 'jenis': 'Kontrak'},
    {'text': 'Kontrak Pengadaan Nomor KTR 007 ABT 2025 kontrak pengadaan bahan baku produksi periode satu tahun', 'arah': 'Masuk', 'jenis': 'Kontrak'},
    {'text': 'Perjanjian Kontrak Outsourcing Nomor KTR 008 ABT 2025 kontrak outsourcing jasa keamanan cleaning service', 'arah': 'Masuk', 'jenis': 'Kontrak'},
    {'text': 'Kontrak Distribusi Nomor KTR 009 ABT 2025 perjanjian distribusi produk perusahaan seluruh wilayah Indonesia', 'arah': 'Keluar', 'jenis': 'Kontrak'},
    {'text': 'Kontrak Sewa Mesin Nomor KTR 010 ABT 2025 perjanjian sewa mesin produksi peralatan pabrik', 'arah': 'Masuk', 'jenis': 'Kontrak'},
    # NOTA DINAS
    {'text': 'Nota Dinas Nomor ND 001 ABT 2025 laporan kegiatan divisi IT bulan Januari capaian target rencana kerja', 'arah': 'Keluar', 'jenis': 'Nota Dinas'},
    {'text': 'Nota Dinas Internal Nomor ND 002 ABT 2025 permintaan pengadaan alat tulis kantor kebutuhan operasional', 'arah': 'Keluar', 'jenis': 'Nota Dinas'},
    {'text': 'Nota Dinas Nomor ND 003 ABT 2025 jadwal rapat koordinasi bulanan seluruh kepala divisi', 'arah': 'Keluar', 'jenis': 'Nota Dinas'},
    {'text': 'Nota Dinas Kepegawaian Nomor ND 004 ABT 2025 permohonan cuti tahunan penggantian jadwal shift', 'arah': 'Keluar', 'jenis': 'Nota Dinas'},
    {'text': 'Nota Dinas Nomor ND 005 ABT 2025 laporan keuangan triwulan pertama tahun anggaran 2025', 'arah': 'Keluar', 'jenis': 'Nota Dinas'},
    {'text': 'Nota Dinas Nomor ND 006 ABT 2025 revisi struktur organisasi penambahan divisi baru', 'arah': 'Keluar', 'jenis': 'Nota Dinas'},
    {'text': 'Nota Dinas Nomor ND 007 ABT 2025 pelaksanaan pelatihan karyawan peningkatan kompetensi', 'arah': 'Keluar', 'jenis': 'Nota Dinas'},
    {'text': 'Nota Dinas Nomor ND 008 ABT 2025 evaluasi kinerja karyawan semester pertama tahun 2025', 'arah': 'Keluar', 'jenis': 'Nota Dinas'},
    {'text': 'Nota Dinas Nomor ND 009 ABT 2025 permintaan perbaikan fasilitas kantor gedung', 'arah': 'Keluar', 'jenis': 'Nota Dinas'},
    {'text': 'Nota Dinas Nomor ND 010 ABT 2025 pengumuman libur nasional cuti bersama tahun 2025', 'arah': 'Keluar', 'jenis': 'Nota Dinas'},
    # MoU
    {'text': 'Memorandum of Understanding Nomor MoU 001 ABT 2025 nota kesepahaman kerjasama PT Almex Universitas Pamulang', 'arah': 'Masuk', 'jenis': 'MoU'},
    {'text': 'MoU Kerjasama Riset Nomor MoU 002 ABT 2025 nota kesepahaman penelitian pengembangan teknologi bersama', 'arah': 'Masuk', 'jenis': 'MoU'},
    {'text': 'Memorandum of Understanding Nomor MoU 003 ABT 2025 kesepahaman kerjasama pelatihan magang mahasiswa', 'arah': 'Masuk', 'jenis': 'MoU'},
    {'text': 'MoU Kemitraan Strategis Nomor MoU 004 ABT 2025 nota kesepahaman kemitraan pengembangan produk inovatif', 'arah': 'Masuk', 'jenis': 'MoU'},
    {'text': 'Memorandum of Understanding Nomor MoU 005 ABT 2025 kesepahaman kerjasama bidang teknologi informasi digitalisasi', 'arah': 'Masuk', 'jenis': 'MoU'},
    {'text': 'MoU Kerjasama Pendidikan Nomor MoU 006 ABT 2025 nota kesepahaman program beasiswa pengembangan SDM', 'arah': 'Masuk', 'jenis': 'MoU'},
    {'text': 'Memorandum of Understanding Nomor MoU 007 ABT 2025 kesepahaman kerjasama pemasaran distribusi regional', 'arah': 'Keluar', 'jenis': 'MoU'},
    {'text': 'MoU Kerjasama Logistik Nomor MoU 008 ABT 2025 nota kesepahaman pengelolaan rantai pasok gudang', 'arah': 'Masuk', 'jenis': 'MoU'},
    {'text': 'Memorandum of Understanding Nomor MoU 009 ABT 2025 kesepahaman kerjasama pengembangan aplikasi mobile', 'arah': 'Masuk', 'jenis': 'MoU'},
    {'text': 'MoU Kerjasama Keamanan Nomor MoU 010 ABT 2025 nota kesepahaman keamanan siber perlindungan data', 'arah': 'Masuk', 'jenis': 'MoU'},
    # SURAT MASUK UMUM
    {'text': 'Surat Undangan Rapat Nomor 001 UND VI 2025 mengundang perwakilan PT Almex rapat koordinasi regional', 'arah': 'Masuk', 'jenis': 'Lainnya'},
    {'text': 'Surat Permohonan Nomor 002 PMH VI 2025 permohonan bantuan teknis pengembangan sistem informasi daerah', 'arah': 'Masuk', 'jenis': 'Lainnya'},
    {'text': 'Surat Pemberitahuan Nomor 003 PBT VI 2025 pemberitahuan perubahan jadwal pelaksanaan tender pengadaan', 'arah': 'Masuk', 'jenis': 'Lainnya'},
    {'text': 'Surat Permintaan Nomor 004 PRM VI 2025 permintaan data informasi keperluan audit tahunan', 'arah': 'Masuk', 'jenis': 'Lainnya'},
    {'text': 'Surat Rekomendasi Nomor 005 RKD VI 2025 rekomendasi dinas terkait pengembangan usaha', 'arah': 'Masuk', 'jenis': 'Lainnya'},
    # SURAT KELUAR UMUM
    {'text': 'Surat Keputusan Nomor 001 SK VI 2025 keputusan direksi pengangkatan karyawan tetap divisi produksi', 'arah': 'Keluar', 'jenis': 'Lainnya'},
    {'text': 'Surat Edaran Nomor 002 SE VI 2025 edaran kebijakan baru jam kerja sistem shift karyawan', 'arah': 'Keluar', 'jenis': 'Lainnya'},
    {'text': 'Surat Tugas Nomor 003 ST VI 2025 penugasan karyawan pelatihan sertifikasi kompetensi', 'arah': 'Keluar', 'jenis': 'Lainnya'},
    {'text': 'Surat Kuasa Nomor 004 SKR VI 2025 pemberian kuasa direktur keuangan menandatangani kontrak', 'arah': 'Keluar', 'jenis': 'Lainnya'},
    {'text': 'Surat Pemberitahuan Nomor 005 SPB VI 2025 pemberitahuan perubahan alamat kantor kontak perusahaan', 'arah': 'Keluar', 'jenis': 'Lainnya'},
]

df = pd.DataFrame(data)
print(f'Total dokumen: {len(df)}')
print()
print('Distribusi Arah Dokumen:')
print(df['arah'].value_counts().to_string())
print()
print('Distribusi Jenis Dokumen:')
print(df['jenis'].value_counts().to_string())
df.head(10)

## 3. Preprocessing Teks
Tahapan: Case Folding → Cleaning → Tokenizing → Stopword Removal → Stemming (Sastrawi)

In [ ]:
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.createStemmer()

stopword_factory = StopWordRemoverFactory()
stopword_list = stopword_factory.getStopWords()

custom_stopwords = set(stopword_list) | {
    'pt', 'cv', 'tbk', 'abt', 'vi', '2025',
    'nomor', 'perihal', 'lampiran', 'kepada', 'yth',
    'yang', 'dan', 'di', 'dengan', 'untuk', 'pada', 'dari',
    'ini', 'itu', 'adalah', 'ke', 'oleh', 'sebagai', 'juga',
    'akan', 'telah', 'sudah', 'atau', 'dalam', 'tidak',
    'ada', 'dapat', 'bisa', 'lebih',
}

def preprocess_text(text):
    text = text.lower()                                  # Case Folding
    text = re.sub(r'[^a-z\s]', ' ', text)               # Cleaning
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()                                # Tokenizing
    tokens = [t for t in tokens if t not in custom_stopwords and len(t) > 2]  # Stopword Removal
    tokens = stemmer.stem(' '.join(tokens)).split()      # Stemming (Sastrawi)
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(preprocess_text)

print('Contoh hasil preprocessing:')
print('=' * 60)
for i in [0, 20, 40, 60]:
    print(f'\nOriginal : {df.iloc[i]["text"][:80]}...')
    print(f'Cleaned  : {df.iloc[i]["clean_text"][:80]}...')
    print(f'Arah: {df.iloc[i]["arah"]}  |  Jenis: {df.iloc[i]["jenis"]}')

## 4. Ekstraksi Fitur TF-IDF

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

X = tfidf_vectorizer.fit_transform(df['clean_text'])
y_arah = df['arah']
y_jenis = df['jenis']

print(f'Shape matriks TF-IDF: {X.shape}')
print(f'Jumlah fitur: {len(tfidf_vectorizer.get_feature_names_out())}')
print()
print('Top 20 fitur TF-IDF tertinggi:')
feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_sum = X.sum(axis=0).A1
top_indices = tfidf_sum.argsort()[-20:][::-1]
for idx in top_indices:
    print(f'  {feature_names[idx]:25s} -> bobot: {tfidf_sum[idx]:.4f}')

## 5. Split Data (80% Training, 20% Testing)

In [ ]:
X_train_arah, X_test_arah, y_train_arah, y_test_arah = train_test_split(
    X, y_arah, test_size=0.2, random_state=42, stratify=y_arah
)

X_train_jenis, X_test_jenis, y_train_jenis, y_test_jenis = train_test_split(
    X, y_jenis, test_size=0.2, random_state=42, stratify=y_jenis
)

print(f'Total data     : {X.shape[0]} dokumen')
print(f'Training (80%) : {X_train_arah.shape[0]} dokumen')
print(f'Testing (20%)  : {X_test_arah.shape[0]} dokumen')
print()
print('Distribusi Arah (Training):')
print(y_train_arah.value_counts().to_string())
print()
print('Distribusi Jenis (Training):')
print(y_train_jenis.value_counts().to_string())

## 6. Training Model - Klasifikasi Arah Dokumen

In [ ]:
model_arah = MultinomialNB(alpha=1.0)  # Laplace Smoothing
model_arah.fit(X_train_arah, y_train_arah)

y_pred_arah = model_arah.predict(X_test_arah)

print('KLASIFIKASI ARAH DOKUMEN (Masuk/Keluar)')
print('=' * 50)
print(f'Accuracy : {accuracy_score(y_test_arah, y_pred_arah):.4f}')
print(f'Precision: {precision_score(y_test_arah, y_pred_arah, average="weighted"):.4f}')
print(f'Recall   : {recall_score(y_test_arah, y_pred_arah, average="weighted"):.4f}')
print(f'F1-Score : {f1_score(y_test_arah, y_pred_arah, average="weighted"):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test_arah, y_pred_arah))

## 7. Confusion Matrix - Arah Dokumen

In [ ]:
cm_arah = confusion_matrix(y_test_arah, y_pred_arah, labels=model_arah.classes_)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_arah, annot=True, fmt='d', cmap='Blues',
            xticklabels=model_arah.classes_, yticklabels=model_arah.classes_,
            linewidths=0.5, linecolor='gray')
plt.title('Confusion Matrix - Klasifikasi Arah Dokumen', fontsize=14, fontweight='bold')
plt.xlabel('Prediksi', fontsize=12)
plt.ylabel('Aktual', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix_arah.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan: confusion_matrix_arah.png')

## 8. Training Model - Klasifikasi Jenis Dokumen

In [ ]:
model_jenis = MultinomialNB(alpha=1.0)
model_jenis.fit(X_train_jenis, y_train_jenis)

y_pred_jenis = model_jenis.predict(X_test_jenis)

print('KLASIFIKASI JENIS DOKUMEN')
print('=' * 50)
print(f'Accuracy : {accuracy_score(y_test_jenis, y_pred_jenis):.4f}')
print(f'Precision: {precision_score(y_test_jenis, y_pred_jenis, average="weighted"):.4f}')
print(f'Recall   : {recall_score(y_test_jenis, y_pred_jenis, average="weighted"):.4f}')
print(f'F1-Score : {f1_score(y_test_jenis, y_pred_jenis, average="weighted"):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test_jenis, y_pred_jenis))

## 9. Confusion Matrix - Jenis Dokumen

In [ ]:
cm_jenis = confusion_matrix(y_test_jenis, y_pred_jenis, labels=model_jenis.classes_)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_jenis, annot=True, fmt='d', cmap='Oranges',
            xticklabels=model_jenis.classes_, yticklabels=model_jenis.classes_,
            linewidths=0.5, linecolor='gray')
plt.title('Confusion Matrix - Klasifikasi Jenis Dokumen', fontsize=14, fontweight='bold')
plt.xlabel('Prediksi', fontsize=12)
plt.ylabel('Aktual', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix_jenis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan: confusion_matrix_jenis.png')

## 10. Cross-Validation (5-Fold)

In [ ]:
cv_arah = cross_val_score(MultinomialNB(alpha=1.0), X, y_arah, cv=5, scoring='accuracy')
cv_jenis = cross_val_score(MultinomialNB(alpha=1.0), X, y_jenis, cv=5, scoring='accuracy')

print('Hasil 5-Fold Cross-Validation:')
print('=' * 50)
print(f'\nArah Dokumen:')
for i, score in enumerate(cv_arah, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'  Mean : {cv_arah.mean():.4f}')
print(f'  Std  : {cv_arah.std():.4f}')

print(f'\nJenis Dokumen:')
for i, score in enumerate(cv_jenis, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'  Mean : {cv_jenis.mean():.4f}')
print(f'  Std  : {cv_jenis.std():.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(range(1, 6), cv_arah, color='#3B82F6', alpha=0.8)
axes[0].axhline(y=cv_arah.mean(), color='red', linestyle='--', label=f'Mean: {cv_arah.mean():.4f}')
axes[0].set_title('Cross-Validation - Arah Dokumen', fontweight='bold')
axes[0].set_xlabel('Fold'); axes[0].set_ylabel('Accuracy'); axes[0].set_ylim(0, 1.1); axes[0].legend()

axes[1].bar(range(1, 6), cv_jenis, color='#F59E0B', alpha=0.8)
axes[1].axhline(y=cv_jenis.mean(), color='red', linestyle='--', label=f'Mean: {cv_jenis.mean():.4f}')
axes[1].set_title('Cross-Validation - Jenis Dokumen', fontweight='bold')
axes[1].set_xlabel('Fold'); axes[1].set_ylabel('Accuracy'); axes[1].set_ylim(0, 1.1); axes[1].legend()

plt.tight_layout()
plt.savefig('cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gambar disimpan: cross_validation.png')

## 11. Simpan Model

In [ ]:
import joblib

joblib.dump(model_arah, '../backend/ml_model/arah_pipeline.pkl')
joblib.dump(model_jenis, '../backend/ml_model/jenis_pipeline.pkl')
joblib.dump(tfidf_vectorizer, '../backend/ml_model/tfidf_vectorizer.pkl')

print('Model berhasil disimpan!')
print('  -> backend/ml_model/arah_pipeline.pkl')
print('  -> backend/ml_model/jenis_pipeline.pkl')
print('  -> backend/ml_model/tfidf_vectorizer.pkl')

## 12. Test Prediksi Dokumen Baru

In [ ]:
test_docs = [
    'Purchase Order Nomor 011 ABT 2025 pemesanan peralatan laboratorium divisi riset',
    'Invoice Tagihan Nomor INV 011 ABT 2025 tagihan jasa konsultasi hukum bulanan',
    'Surat Penawaran Kerjasama Nomor SP 011 ABT 2025 penawaran kerjasama pengembangan aplikasi',
    'Nota Dinas Nomor ND 011 ABT 2025 evaluasi kinerja divisi pemasaran triwulan kedua',
    'Memorandum of Understanding Nomor MoU 011 ABT 2025 kesepahaman kerjasama riset kecerdasan buatan',
]

print('PREDIKSI DOKUMEN BARU')
print('=' * 70)

for i, doc in enumerate(test_docs, 1):
    clean = preprocess_text(doc)
    vec = tfidf_vectorizer.transform([clean])

    arah = model_arah.predict(vec)[0]
    arah_conf = max(model_arah.predict_proba(vec)[0]) * 100

    jenis = model_jenis.predict(vec)[0]
    jenis_conf = max(model_jenis.predict_proba(vec)[0]) * 100

    print(f'\nDokumen {i}: {doc[:60]}...')
    print(f'  -> Arah  : {arah} (confidence: {arah_conf:.1f}%)')
    print(f'  -> Jenis : {jenis} (confidence: {jenis_conf:.1f}%)')